# Twinkle 客户端 - 自我认知 LoRA 训练

本 Notebook 演示如何通过 **Twinkle 客户端** 使用 LoRA 微调语言模型，使其学会自定义身份。

### Twinkle 客户端 vs Tinker 客户端

Twinkle 提供两种客户端 API：
- **Tinker 客户端**（见 `tinker/notebook/`）：更底层，手动控制 forward/backward/optim
- **Twinkle 客户端**（本 Notebook）：更高层，提供 `Model` 抽象，自动管理梯度累积、优化器、检查点等

### 整体流程

```
初始化客户端 -> 查询历史训练 -> 准备数据集 -> 配置模型/LoRA/优化器 -> 训练循环 -> 保存检查点
```

### 前置条件

| 条件 | 说明 |
|------|------|
| 服务端已启动 | 参见 `cookbook/client/server/` 下的配置和启动脚本 |
| 环境变量 | 在 `.env` 文件或环境中设置 `MODELSCOPE_TOKEN` |
| 依赖 | `pip install dotenv peft twinkle twinkle_client` |

## 第一步：导入依赖并加载环境变量

使用 `dotenv` 从 `.env` 文件加载 API Token 等配置。如果你直接设置了环境变量，也可以跳过 dotenv。

In [ ]:
import dotenv
dotenv.load_dotenv('.env')

import os
from peft import LoraConfig

from twinkle import get_logger
from twinkle.dataset import DatasetMeta
from twinkle_client import init_twinkle_client
from twinkle_client.dataloader import DataLoader
from twinkle_client.dataset import Dataset
from twinkle_client.model import MultiLoraTransformersModel

logger = get_logger()

## 第二步：初始化客户端并连接服务端

| 参数 | 说明 |
|------|------|
| `base_url` | 服务端地址，ModelScope 官方环境为 `http://www.modelscope.cn/twinkle` |
| `api_key` | ModelScope Token，用于身份认证 |
| `base_model` | 基座模型名称 |

In [ ]:
base_model = 'Qwen/Qwen3.5-27B'
base_url = 'http://www.modelscope.cn/twinkle'

client = init_twinkle_client(base_url=base_url, api_key=os.environ.get('MODELSCOPE_TOKEN'))

## 第三步：查询历史训练记录（可选）

Twinkle 客户端支持查询服务端上已有的训练记录和检查点，方便 **断点续训**。

如果你是第一次训练，这里会返回空列表。如果想从某个检查点恢复，取消注释 `resume_path = checkpoint.twinkle_path`。

In [ ]:
runs = client.list_training_runs()

resume_path = None
for run in runs:
    logger.info(run.model_dump_json(indent=2))
    checkpoints = client.list_checkpoints(run.training_run_id)
    for checkpoint in checkpoints:
        logger.info(checkpoint.model_dump_json(indent=2))
        # 如需断点续训，取消下面的注释：
        # resume_path = checkpoint.twinkle_path

print(f'Found {len(runs)} training run(s), resume_path={resume_path}')

## 第四步：准备自我认知数据集

处理流程与 Tinker 版本类似：
1. 从 ModelScope 加载 `swift/self-cognition` 数据集（前 500 条）
2. 应用 chat template（最大 512 token）
3. 用 `SelfCognitionProcessor` 替换身份占位符
4. 编码为 token 序列

> **注意**：这里使用的是 `twinkle_client.dataset.Dataset`（而非 `twinkle.dataset.Dataset`），它是专为 Twinkle 客户端设计的封装。

In [ ]:
dataset = Dataset(dataset_meta=DatasetMeta('ms://swift/self-cognition', data_slice=range(500)))

# 应用 chat template
dataset.set_template('Template', model_id=f'ms://{base_model}', max_length=512)

# 替换身份：model_name 是模型名称，model_author 是团队名称
dataset.map('SelfCognitionProcessor', init_args={'model_name': 'twinkle模型', 'model_author': 'ModelScope社区'})

# 编码
dataset.encode(batched=True)

dataloader = DataLoader(dataset=dataset, batch_size=4)
print(f'Dataset size: {len(dataset)}')

## 第五步：配置模型

Twinkle 客户端使用 `MultiLoraTransformersModel` 管理模型，支持多个 LoRA 适配器同时加载。

配置项逐一说明：

| 配置 | 方法 | 说明 |
|------|------|------|
| LoRA 适配器 | `add_adapter_to_model` | 添加 LoRA，`target_modules='all-linear'` 对所有线性层加适配器 |
| 梯度累积 | `gradient_accumulation_steps=2` | 每 2 个 micro-batch 更新一次，等效 batch_size=8 |
| 模板 | `set_template` | 与数据预处理使用相同模板 |
| 输入处理 | `set_processor` | padding_side='right' 右侧填充 |
| 损失函数 | `set_loss` | 交叉熵损失 |
| 优化器 | `set_optimizer` | Adam，lr=1e-4 |

> **提示**：如果服务端使用 Megatron 后端，不支持 LR scheduler，所以此处注释掉了 `set_lr_scheduler`。

In [ ]:
model = MultiLoraTransformersModel(model_id=f'ms://{base_model}')

# LoRA 配置
lora_config = LoraConfig(target_modules='all-linear')
model.add_adapter_to_model('default', lora_config, gradient_accumulation_steps=2)

# 模板和处理器
model.set_template('Template')
model.set_processor('InputProcessor', padding_side='right')

# 损失函数和优化器
model.set_loss('CrossEntropyLoss')
model.set_optimizer('Adam', lr=1e-4)

# LR Scheduler（Megatron 后端不支持，按需开启）
# model.set_lr_scheduler('LinearLR')

print('Model configured')

## 第六步：断点续训（可选）

如果第三步中找到了可恢复的检查点，这里会加载模型权重和优化器状态。

In [ ]:
if resume_path:
    logger.info(f'Resuming training from {resume_path}')
    model.load(resume_path, load_optimizer=True)
else:
    print('Starting fresh training (no checkpoint to resume)')

logger.info(model.get_train_configs().model_dump())

## 第七步：训练循环

每个 step 的流程：
1. `forward_backward`：前向 + 反向传播
2. `clip_grad_and_step`：梯度裁剪 + 优化器更新 + 梯度清零（一步完成）
3. 每 2 步打印一次指标（与 gradient_accumulation_steps 对齐）
4. 每个 epoch 结束保存检查点

| 参数 | 值 | 说明 |
|------|-----|------|
| epoch 数 | 3 | 训练轮数 |
| batch_size | 4 | 每 batch 样本数 |
| 有效 batch | 8 | batch_size x gradient_accumulation_steps |

> `clip_grad_and_step()` 等价于依次调用 `clip_grad_norm(1.0)` → `step()` → `zero_grad()` → `lr_step()`。

In [ ]:
for epoch in range(3):
    logger.info(f'Starting epoch {epoch}')
    for step, batch in enumerate(dataloader):
        # 前向 + 反向
        model.forward_backward(inputs=batch)

        # 梯度裁剪 + 优化器更新
        model.clip_grad_and_step()

        # 每 2 步打印指标
        if step % 2 == 0:
            metric = model.calculate_metric(is_training=True)
            logger.info(f'Current is step {step} of {len(dataloader)}, metric: {metric.result}')

    # 保存检查点
    twinkle_path = model.save(name=f'twinkle-epoch-{epoch}', save_optimizer=True)
    logger.info(f'Saved checkpoint: {twinkle_path}')

## 第八步：上传到 ModelScope Hub（可选）

训练完成后，可以将检查点上传到 ModelScope Hub 进行分享。

取消注释并填写你的用户名即可使用。

In [ ]:
# YOUR_USER_NAME = "your_username"
# hub_model_id = f'{YOUR_USER_NAME}/twinkle-self-cognition'
# model.upload_to_hub(
#     checkpoint_dir=twinkle_path,
#     hub_model_id=hub_model_id,
#     async_upload=False
# )
# logger.info(f'Uploaded checkpoint to hub: {hub_model_id}')

print('Training complete!')

## 常见问题

| 问题 | 可能原因 | 解决方法 |
|------|----------|----------|
| `init_twinkle_client` 连接失败 | 服务端未启动或地址错误 | 确认服务端正常运行，检查 base_url |
| `.env` 文件未生效 | 文件路径不正确 | 确保 `.env` 在工作目录下，或直接设置环境变量 |
| Loss 不下降 | 学习率不合适 | 尝试 5e-5 或 2e-4 |
| 断点续训失败 | 检查点路径无效 | 通过 `list_checkpoints` 确认可用路径 |
| Megatron 后端 LR scheduler 报错 | 不支持 | 注释掉 `set_lr_scheduler` |